## Task 2: Recursive utility functions

In [ ]:
from pathlib import Path
import json


def find_log_files(path, extension='.log'):
    """Recursively find all files with the given extension below path."""
    path = Path(path)
    found = []
    for item in path.iterdir():
        if item.is_dir():
            # Recursive step: search inside the subdirectory
            found.extend(find_log_files(item, extension))
        elif item.is_file() and item.suffix == extension:
            found.append(item)
    # Base case: when a directory has no more subdirectories, the look ends
    return found


log_files = find_log_files('data')
print(len(log_files))
log_files[:3]

9


[WindowsPath('data/2026/library_cafe/POS_01/register_sales.log'),
 WindowsPath('data/2026/library_cafe/POS_02/register_sales.log'),
 WindowsPath('data/2026/library_cafe/POS_03/register_sales.log')]

In [ ]:
def get_active_registers(node) -> set[str]:
    """Recursively collect register ids where "active" is true."""
    active = set()
    if isinstance(node, dict):
        # A register is a dict with "active": true and an "id"
        if node.get('active') is True and 'id' in node:
            active.add(node['id'])
        # Keep searching deeper in all values of the dict
        for value in node.values():
            active |= get_active_registers(value)
    elif isinstance(node, list):
        # Lists can contain dicts or more lists, so recurse on each item
        for item in node:
            active |= get_active_registers(item)
    # Strings, numbers etc. fall through and return the empty set
    return active


with open('data/store_config.json', encoding='utf-8') as f:
    config = json.load(f)

active_registers = get_active_registers(config)
print(active_registers)

{'POS_02', 'POS_01', 'POS_03'}


## Task 3: Cleaning and analysing data with NumPy

In [ ]:
import numpy as np


def remove_outliers(prices) -> np.ndarray:
    """Replace values more than 3 standard deviations from the mean
    with the median of the valid prices."""
    prices = np.asarray(prices, dtype=float).ravel()
    mean = prices.mean()
    std = prices.std()
    is_outlier = np.abs(prices - mean) > 3 * std
    median = np.median(prices[~is_outlier])
    cleaned = prices.copy()
    cleaned[is_outlier] = median
    return cleaned


test = np.array([4.5, 3.0, 4.2, 3.5, 3.8, 7.5, 4.0, 3.2, 4.4, 3.9, 4.1, 3.6, 999.0])
print(remove_outliers(test))

[4.5  3.   4.2  3.5  3.8  7.5  4.   3.2  4.4  3.9  4.1  3.6  3.95]


In [12]:
def compute_refund_ratio(sales_count: np.ndarray, refund_count: np.ndarray) -> np.ndarray:
    """Element-wise refund ratio: refund / (sales + 1e-6)."""
    sales_count = np.asarray(sales_count, dtype=float)
    refund_count = np.asarray(refund_count, dtype=float)
    return refund_count / (sales_count + 1e-6)


print(compute_refund_ratio(np.array([100, 50, 0]), np.array([5, 0, 0])))

[0.05 0.   0.  ]


## Task 4: Data analysis with pandas

In [13]:
import pandas as pd


def build_dataframe(records: list[dict]) -> pd.DataFrame:
    """Records -> DataFrame with datetime index, sorted by time."""
    df = pd.DataFrame(records)
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    return df.set_index('timestamp').sort_index()


def hourly_revenue(df: pd.DataFrame) -> pd.DataFrame:
    """Total price per item in 1-hour windows."""
    return df.groupby('item')['price'].resample('1h').sum().reset_index()


def status_pivot(df: pd.DataFrame) -> pd.DataFrame:
    """Total revenue per item across status categories."""
    pivot = df.pivot_table(index='item', columns='status', values='price',
                           aggfunc='sum', fill_value=0)
    return pivot.reindex(columns=['SUCCESS', 'REFUND', 'CANCELLED'], fill_value=0)


def item_refund_ratios(df: pd.DataFrame) -> pd.DataFrame:
    """Refund ratio per item, using compute_refund_ratio from Task 3."""
    counts = df.groupby(['item', 'status']).size().unstack(fill_value=0)
    counts = counts.reindex(columns=['SUCCESS', 'REFUND', 'CANCELLED'], fill_value=0)
    ratio = compute_refund_ratio(counts['SUCCESS'].to_numpy(),
                                 counts['REFUND'].to_numpy())
    return pd.DataFrame({'item': counts.index, 'refund_ratio': ratio})

In [14]:
fake = [
    {'timestamp': '2026-10-15 08:30:00', 'register_id': 'POS_01', 'item': 'Latte', 'price': 4.5, 'currency': 'EUR', 'status': 'SUCCESS'},
    {'timestamp': '2026-10-15 08:45:00', 'register_id': 'POS_01', 'item': 'Latte', 'price': 4.5, 'currency': 'EUR', 'status': 'REFUND'},
    {'timestamp': '2026-10-15 09:10:00', 'register_id': 'POS_02', 'item': 'Muffin', 'price': 3.5, 'currency': 'EUR', 'status': 'SUCCESS'},
    {'timestamp': '2026-10-15 09:20:00', 'register_id': 'POS_02', 'item': 'Latte', 'price': 4.5, 'currency': 'EUR', 'status': 'SUCCESS'},
]
df_test = build_dataframe(fake)
print(hourly_revenue(df_test))
print(status_pivot(df_test))
print(item_refund_ratios(df_test))

     item           timestamp  price
0   Latte 2026-10-15 08:00:00    9.0
1   Latte 2026-10-15 09:00:00    4.5
2  Muffin 2026-10-15 09:00:00    3.5
status  SUCCESS  REFUND  CANCELLED
item                              
Latte       9.0     4.5          0
Muffin      3.5     0.0          0
     item  refund_ratio
0   Latte           0.5
1  Muffin           0.0
